# MAPPO Learning Curves

Visualises training metrics produced by `scripts/train_mappo.py`.
Reads SB3 TensorBoard event files **or** progress CSV files from
`logs/mappo_dmpc/` and plots reward, entropy, and loss curves.

**Run training first:**
```bash
python scripts/train_mappo.py --config config/mappo_config.yaml
```

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11})

LOG_DIR = Path('../logs/mappo_dmpc')
print('Log directory:', LOG_DIR.resolve())
print('Exists:', LOG_DIR.exists())
if LOG_DIR.exists():
    entries = sorted(LOG_DIR.iterdir())
    print('Contents:', [e.name for e in entries])

In [ ]:
# ── Load TensorBoard scalars (if tensorboard is installed) ─────────
# Falls back to SB3 progress CSV when TB is unavailable.

def load_tb_scalars(log_dir: Path) -> dict[str, np.ndarray]:
    """Return {tag: values_array} from the first TF event file found."""
    try:
        from tensorboard.backend.event_processing.event_accumulator import (
            EventAccumulator,
        )
        ea = EventAccumulator(str(log_dir))
        ea.Reload()
        tags = ea.Tags().get('scalars', [])
        return {t: np.array([s.value for s in ea.Scalars(t)]) for t in tags}
    except Exception as exc:
        print(f'TensorBoard not available or no events found: {exc}')
        return {}

def load_sb3_csv(log_dir: Path) -> dict[str, np.ndarray]:
    """Return scalar arrays from SB3 monitor CSV."""
    try:
        import pandas as pd
        csvs = sorted(log_dir.rglob('*.csv'))
        if not csvs:
            return {}
        df = pd.concat([pd.read_csv(p, comment='#') for p in csvs], ignore_index=True)
        return {col: df[col].values for col in df.columns}
    except Exception as exc:
        print(f'CSV load failed: {exc}')
        return {}

# Try TensorBoard first, then CSV
scalars = {}
if LOG_DIR.exists():
    # Look for a MAPPOAgent subdirectory
    tb_dirs = [d for d in LOG_DIR.rglob('events.out.tfevents.*')]
    if tb_dirs:
        scalars = load_tb_scalars(str(tb_dirs[0].parent))
    if not scalars:
        scalars = load_sb3_csv(LOG_DIR)

if scalars:
    print('Loaded tags:', list(scalars.keys())[:10])
else:
    print('No training data found — run train_mappo.py first.')
    # Synthesise dummy data for notebook illustration
    N = 500
    t = np.linspace(0, 1, N)
    scalars = {
        'rollout/ep_rew_mean': -50 + 60 * t + 5 * np.random.randn(N),
        'train/entropy_loss':  -1.5 + 0.5 * t + 0.05 * np.random.randn(N),
        'train/value_loss':     2.0 * np.exp(-3 * t) + 0.02 * np.random.randn(N),
        'train/policy_gradient_loss': -0.02 * np.ones(N) + 0.005 * np.random.randn(N),
    }
    print('Using synthetic data for illustration.')

In [ ]:
def smooth(arr, w=20):
    kernel = np.ones(w) / w
    return np.convolve(arr, kernel, mode='same')

reward_key = next((k for k in scalars if 'rew_mean' in k or 'reward' in k.lower()), None)
entropy_key = next((k for k in scalars if 'entropy' in k.lower()), None)
value_key   = next((k for k in scalars if 'value_loss' in k.lower()), None)
pg_key      = next((k for k in scalars if 'policy_gradient' in k.lower()), None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if reward_key:
    r = scalars[reward_key]
    x = np.arange(len(r))
    axes[0].plot(x, r, alpha=0.3, color='steelblue', label='Episode reward')
    axes[0].plot(x, smooth(r), color='navy', lw=2, label='Smoothed')
    axes[0].set_xlabel('Update step')
    axes[0].set_ylabel('Mean episode reward')
    axes[0].set_title('MAPPO Training — Episode Reward', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

if entropy_key:
    e = scalars[entropy_key]
    x = np.arange(len(e))
    axes[1].plot(x, e, alpha=0.3, color='tomato', label='Entropy loss')
    axes[1].plot(x, smooth(e), color='darkred', lw=2, label='Smoothed')
    axes[1].set_xlabel('Update step')
    axes[1].set_ylabel('Entropy loss')
    axes[1].set_title('MAPPO Training — Policy Entropy', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved learning_curves.png')

In [ ]:
# ── Value-function and policy-gradient loss ────────────────────────
if value_key or pg_key:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if value_key:
        v = scalars[value_key]
        x = np.arange(len(v))
        axes[0].plot(x, v, alpha=0.3, color='mediumpurple')
        axes[0].plot(x, smooth(v), color='purple', lw=2)
        axes[0].set_title('Value-Function Loss', fontweight='bold')
        axes[0].set_xlabel('Update step')
        axes[0].grid(alpha=0.3)

    if pg_key:
        pg = scalars[pg_key]
        x  = np.arange(len(pg))
        axes[1].plot(x, pg, alpha=0.3, color='mediumseagreen')
        axes[1].plot(x, smooth(pg), color='darkgreen', lw=2)
        axes[1].set_title('Policy-Gradient Loss', fontweight='bold')
        axes[1].set_xlabel('Update step')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()